# Optimizer Comparison - Fresh Server Run

Designed to run on a fresh GPU server (vast.ai or any CUDA box).

Only edit the `Configuration - EDIT THIS CELL` cell below. The notebook now prints the exact run plan before training: datasets, models, optimizers, HPO trials/configurations, evaluation runs, seeds, and epochs.

Then run it either from Jupyter (`Kernel -> Restart & Run All`) or from tmux with `jupyter nbconvert --execute` as shown in the terminal instructions cell.


## ① Configuration — EDIT THIS CELL

In [ ]:
# ===============================================================================
# EDIT THIS CELL - everything else runs automatically
# ===============================================================================

# GitHub access - needed to clone the private repo.
# Create a token at https://github.com/settings/tokens (scope: repo).
# Leave empty if the repo is public or you already cloned it.
GITHUB_TOKEN = ""  # e.g. "ghp_xxxxxxxxxxxx"

# Where to put everything (code, data, outputs, MLflow SQLite DB).
# Will be created if it doesn't exist. /root is fine for most GPU instances.
BASE_DIR = "/root/optimizer_run"

# Kaggle credentials - only needed for Kaggle-only datasets such as Intel Image.
# Paste the contents of your kaggle.json here as a dict, or leave None.
KAGGLE_CREDENTIALS = None
# KAGGLE_CREDENTIALS = {"username": "you", "key": "abc123"}

# -- Experiment settings --------------------------------------------------------
# smoke      : quick wiring test, 2 configs/optimizer, 2 epochs, 1 seed, mock data
# grid_check : quick image-grid check, 6 configs/optimizer, 2 epochs, 1 seed, mock data
# quality    : real run with task-specific counts below
RUN_PRESET = "quality"

# Run switches. Leave dataset/model lists as [] to run every default experiment
# for the enabled task.
RUN_REGRESSION             = True
RUN_TABULAR_CLASSIFICATION = True
RUN_IMAGE_CLASSIFICATION   = True

# Per-task dataset/model overrides - [] means use TASK_DEFAULTS from src.runner:
#   regression:            datasets=[superconductivity, yearmsd]  models=[simple_mlp, residual_mlp]
#   tabular_classification: datasets=[adult, creditcard]          models=[simple_cls, attention_cls]
#   image_classification:  datasets=[cifar100, intel]             models=[resnet18, efficientnet_v2_s]
REGRESSION_DATASETS = []  # defaults: ["superconductivity", "yearmsd"]
REGRESSION_MODELS   = ["simple_mlp", "residual_mlp"]
TABULAR_DATASETS    = []
TABULAR_MODELS      = []
IMAGE_DATASETS      = []  # defaults: ["cifar100", "intel"]
IMAGE_MODELS        = ["resnet18", "efficientnet_v2_s"]

# Quality preset: explicit task-specific run sizes.
# HPO_CONFIGS_PER_OPTIMIZER is passed to Ray Tune as num_samples, so total HPO
# configs for one dataset/model pair = optimizers * HPO_CONFIGS_PER_OPTIMIZER.
QUALITY_HPO_CONFIGS_PER_OPTIMIZER = {
    "regression": 48,
    "tabular_classification": 48,
    "image_classification": 24,
}
QUALITY_EPOCHS = {
    "regression": 50,
    "tabular_classification": 26,
    "image_classification": 24,
}
QUALITY_SEEDS = [0, 1, 2]

# CIFAR100 converged earlier in previous runs; keep this override explicit.
# Set to None to use QUALITY_EPOCHS["image_classification"] instead.
CIFAR100_NUM_EPOCHS = 10

# Fast preset sizes.
SMOKE_HPO_CONFIGS_PER_OPTIMIZER = 2
SMOKE_EPOCHS = 2
SMOKE_SEEDS = [0]
GRID_CHECK_HPO_CONFIGS_PER_OPTIMIZER = 6
GRID_CHECK_EPOCHS = 2
GRID_CHECK_SEEDS = [0]
GRID_CHECK_MAX_IMGS = 256

NUM_WORKERS = 4  # DataLoader workers per final evaluation run
GRID_CHECK_IMAGE_PARAMS = RUN_PRESET == "grid_check"
MOCK_RUN = RUN_PRESET == "smoke"

if RUN_PRESET not in {"quality", "smoke", "grid_check"}:
    raise ValueError(f"Unknown RUN_PRESET={RUN_PRESET!r}")

if GRID_CHECK_IMAGE_PARAMS:
    RUN_REGRESSION = False
    RUN_TABULAR_CLASSIFICATION = False
    RUN_IMAGE_CLASSIFICATION = True


def task_run_settings(task_type, datasets=None):
    """Return task-specific HPO config count, seeds, epochs and mock flag."""
    datasets = datasets or []
    if RUN_PRESET == "smoke":
        return {
            "num_samples": SMOKE_HPO_CONFIGS_PER_OPTIMIZER,
            "seeds": SMOKE_SEEDS,
            "num_epochs": SMOKE_EPOCHS,
            "mock_run": True,
        }
    if RUN_PRESET == "grid_check":
        return {
            "num_samples": GRID_CHECK_HPO_CONFIGS_PER_OPTIMIZER,
            "seeds": GRID_CHECK_SEEDS,
            "num_epochs": GRID_CHECK_EPOCHS,
            "mock_run": False,
        }

    epochs = QUALITY_EPOCHS[task_type]
    if task_type == "image_classification" and datasets == ["cifar100"] and CIFAR100_NUM_EPOCHS is not None:
        epochs = CIFAR100_NUM_EPOCHS
    return {
        "num_samples": QUALITY_HPO_CONFIGS_PER_OPTIMIZER[task_type],
        "seeds": QUALITY_SEEDS,
        "num_epochs": epochs,
        "mock_run": False,
    }

# Backward-compatible globals for any ad-hoc cells below.
_default_quality = task_run_settings("image_classification", IMAGE_DATASETS)
NUM_SAMPLES = _default_quality["num_samples"]
SEEDS = _default_quality["seeds"]
# ===============================================================================


## ② Bootstrap — detect environment, derive paths

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

BASE_DIR  = Path(BASE_DIR).expanduser().resolve()
CODE_DIR  = BASE_DIR / "code"
DATA_DIR  = BASE_DIR / "data"
OUT_DIR   = BASE_DIR / "outputs"
MFLOW_DB  = BASE_DIR / "mlflow.db"
MFLOW_URI = f"sqlite:///{MFLOW_DB}"

for d in (BASE_DIR, CODE_DIR, DATA_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR  : {BASE_DIR}")
print(f"CODE_DIR  : {CODE_DIR}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUT_DIR   : {OUT_DIR}")
print(f"MFLOW_DB  : {MFLOW_DB}")
print(f"MFLOW_URI : {MFLOW_URI}")

# Write Kaggle credentials if provided
if KAGGLE_CREDENTIALS:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    kaggle_json_path = kaggle_dir / "kaggle.json"
    kaggle_json_path.write_text(json.dumps(KAGGLE_CREDENTIALS))
    kaggle_json_path.chmod(0o600)
    KAGGLE_JSON = str(kaggle_json_path)
    print(f"kaggle.json written to {kaggle_json_path}")
else:
    KAGGLE_JSON = None

print("Paths OK")

## ③ Install system packages & Python dependencies

In [ ]:
def _run(cmd, **kw):
    """Run a shell command, stream output, raise on error."""
    print(f"$ {' '.join(cmd) if isinstance(cmd, list) else cmd}")
    result = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if result.stdout.strip():
        print(result.stdout[-3000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# System packages (apt)
apt_pkgs = ["git", "curl", "rsync"]
missing_apt = [p for p in apt_pkgs if not shutil.which(p)]
if missing_apt:
    _run(["apt-get", "install", "-y", "--no-install-recommends"] + missing_apt)
else:
    print("apt packages already present:", apt_pkgs)

print("System deps OK")

## ④ Clone the project repository

In [ ]:
REPO = "github.com/maximspbu/optimizers_comparison_tracking.git"
BRANCH = "main"

if (CODE_DIR / "src").exists():
    print(f"Code already present at {CODE_DIR}, pulling latest ...")
    _run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"])
else:
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@{REPO}"
    else:
        clone_url = f"https://{REPO}"  # works for public repos
    print(f"Cloning branch '{BRANCH}' ...")
    _run(["git", "clone", "--depth=1", "--branch", BRANCH, clone_url, str(CODE_DIR)])

# Add code dir to path so `import src` works
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.chdir(str(CODE_DIR))

print(f"cwd: {os.getcwd()}")
print("Repo OK")

In [ ]:
# Install Python dependencies
req_file = CODE_DIR / "requirements.server.txt"
base_pip = [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages", "--root-user-action=ignore"]
# Debian images can ship blinker without pip RECORD metadata; install a pip-owned copy first.
_run(base_pip + ["--ignore-installed", "blinker"])
# Keep the server's CUDA-compatible torch build and use wheels for Python 3.14 scientific packages.
_run(base_pip + ["--only-binary=:all:", "numpy", "scipy", "pandas", "scikit-learn", "matplotlib", "seaborn", "pillow"])
_run(base_pip + ["-r", str(req_file)])
try:
    import torchvision  # noqa: F401
except ImportError:
    _run(base_pip + ["--no-deps", "torchvision"])
print("pip requirements OK")

In [ ]:
# Clone optimizer repos not on PyPI
import importlib

EXTRA_REPOS = [
    ("https://github.com/nanowell/AdEMAMix-Optimizer-Pytorch", "AdEMAMix_Optimizer_Pytorch"),
]
for url, name in EXTRA_REPOS:
    dest = CODE_DIR / name
    if not dest.exists():
        print(f"Cloning {name} ...")
        _run(["git", "clone", "--depth=1", url, str(dest)])
    else:
        print(f"{name} already present")

# Keep CODE_DIR first so project imports resolve as src.*.
# Stacey++ is vendored locally in src/optimizers.py, so no external Stacey clone is needed.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
print("Extra repos OK")

## ⑤ Environment verification

In [ ]:
import importlib

import torch
print(f"torch {torch.__version__}  CUDA: {torch.cuda.is_available()}")
GPU_NUM = torch.cuda.device_count()
for i in range(GPU_NUM):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB")
if GPU_NUM == 0:
    print("  No GPU detected — will run on CPU (slow!)")
    GPU_NUM = 0

for pkg in ("pytorch_lightning", "ray", "mlflow", "optuna", "openml"):
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: {getattr(m, '__version__', 'ok')}")
    except ImportError:
        print(f"  {pkg}: MISSING — re-run the pip cell above")

optimizer_imports = {
    "AdamW": ("torch.optim", "AdamW"),
    "Stacey_pp": ("src.optimizers", "Stacey_pp"),
    "Lion": ("lion_pytorch", "Lion"),
    "AdamWScheduleFree": ("schedulefree", "AdamWScheduleFree"),
    "GaLoreAdamW": ("src.optimizers", "GaLoreAdamW"),
    "AdEMAMix": ("AdEMAMix_Optimizer_Pytorch.AdEMAMix", "AdEMAMix"),
}
missing_optimizers = []
for opt_name, (module_name, attr_name) in optimizer_imports.items():
    try:
        module = importlib.import_module(module_name)
        optimizer_cls = getattr(module, attr_name)
        if optimizer_cls is None:
            raise ImportError(f"{module_name}.{attr_name} is None")
        print(f"  optimizer {opt_name}: OK")
    except Exception as exc:
        missing_optimizers.append(f"{opt_name} ({module_name}.{attr_name}): {exc!r}")

if missing_optimizers:
    raise RuntimeError("Missing optimizer imports:\n" + "\n".join(missing_optimizers))

import src.config as project_config
project_config = importlib.reload(project_config)
optimizer_sets = {
    "regression": project_config.REGRESSION_OPTIMIZERS_PARAMS,
    "tabular_classification": project_config.CLASSIFICATION_OPTIMIZERS_PARAMS,
    "image_classification": project_config.IMAGE_CLASSIFICATION_OPTIMIZERS_PARAMS,
}
for task_type, optimizer_params in optimizer_sets.items():
    names = project_config.optimizer_names(optimizer_params)
    project_config.validate_required_optimizers(optimizer_params, task_type)
    print(f"  {task_type}: {names}")

print("\nEnvironment check done.")

## ⑥ Configure logging & MLflow

In [ ]:
import logging

log_path = OUT_DIR / "run.log"
fmt = "%(asctime)s  %(levelname)-8s  %(name)s  %(message)s"
logging.basicConfig(
    level=logging.INFO,
    format=fmt,
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(str(log_path), mode="a"),
    ],
    force=True,
)
for noisy in ("ray", "ray.tune", "ray.air", "pytorch_lightning",
              "lightning", "absl", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

import mlflow
mlflow.set_tracking_uri(MFLOW_URI)

import importlib
for module_name in list(sys.modules):
    if module_name in {"src.config", "src.runner"}:
        del sys.modules[module_name]
importlib.invalidate_caches()

from src.runner import ExperimentConfig, run_experiments, TASK_DEFAULTS

print(f"Logging → {log_path}")
print("Ready to run experiments.")

## Run plan - verify counts before training

This cell prints the exact amount of work that will run. `configs/optimizer` is Ray Tune `num_samples`; total HPO configurations are multiplied by the number of optimizers, datasets, and models.


In [ ]:
import src.config as _plan_config

_optimizer_counts = {
    "regression": len(_plan_config.REGRESSION_OPTIMIZERS_PARAMS),
    "tabular_classification": len(_plan_config.CLASSIFICATION_OPTIMIZERS_PARAMS),
    "image_classification": len(_plan_config.IMAGE_CLASSIFICATION_OPTIMIZERS_PARAMS),
}


def _plan_row(task_type, enabled, datasets, models):
    defaults = TASK_DEFAULTS[task_type]
    resolved_datasets = datasets or defaults["datasets"]
    resolved_models = models or defaults["model_types"]
    settings = task_run_settings(task_type, resolved_datasets)
    optimizers = _optimizer_counts[task_type]
    pairs = len(resolved_datasets) * len(resolved_models)
    hpo_configs = pairs * optimizers * settings["num_samples"]
    eval_runs = 0 if GRID_CHECK_IMAGE_PARAMS else pairs * optimizers * len(settings["seeds"]) * 2
    return {
        "enabled": enabled,
        "task": task_type,
        "datasets": resolved_datasets,
        "models": resolved_models,
        "pairs": pairs,
        "optimizers": optimizers,
        "configs/optimizer": settings["num_samples"],
        "total_hpo_configs": hpo_configs,
        "epochs": settings["num_epochs"],
        "seeds": settings["seeds"],
        "tuned_plus_default_eval_runs": eval_runs,
        "mock_run": settings["mock_run"],
    }

_run_plan = [
    _plan_row("regression", RUN_REGRESSION, REGRESSION_DATASETS, REGRESSION_MODELS),
    _plan_row("tabular_classification", RUN_TABULAR_CLASSIFICATION, TABULAR_DATASETS, TABULAR_MODELS),
    _plan_row("image_classification", RUN_IMAGE_CLASSIFICATION, IMAGE_DATASETS, IMAGE_MODELS),
]

print(f"RUN_PRESET={RUN_PRESET!r}  GPU_NUM={GPU_NUM}  NUM_WORKERS={NUM_WORKERS}")
for row in _run_plan:
    status = "RUN" if row["enabled"] else "SKIP"
    print(f"\n[{status}] {row['task']}")
    print(f"  datasets={row['datasets']}")
    print(f"  models={row['models']}")
    print(
        f"  pairs={row['pairs']}  optimizers={row['optimizers']}  "
        f"configs/optimizer={row['configs/optimizer']}  total_hpo_configs={row['total_hpo_configs']}"
    )
    print(
        f"  epochs={row['epochs']}  seeds={row['seeds']}  "
        f"tuned+default_eval_runs={row['tuned_plus_default_eval_runs']}  mock_run={row['mock_run']}"
    )

print("\nProgress files:")
print(f"  runner log: {OUT_DIR / 'results' / 'run.log'}")
print(f"  notebook log: {OUT_DIR / 'run.log'}")
print(f"  Ray results: {OUT_DIR / 'ray_results'}")


## ⑦ Run: Regression

In [ ]:
if RUN_REGRESSION:
    defaults = TASK_DEFAULTS["regression"]
    reg_datasets = REGRESSION_DATASETS or defaults["datasets"]
    reg_settings = task_run_settings("regression", reg_datasets)
    cfg_reg = ExperimentConfig(
        task_type   = "regression",
        datasets    = reg_datasets,
        model_types = REGRESSION_MODELS or defaults["model_types"],
        num_samples = reg_settings["num_samples"],
        seeds       = reg_settings["seeds"],
        batch_size  = defaults["batch_size"],
        num_epochs  = reg_settings["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = reg_settings["mock_run"],
        output_dir  = str(OUT_DIR),
        mlflow_uri  = MFLOW_URI,
        data_dir    = str(DATA_DIR),
        num_workers = NUM_WORKERS,
    )
    print(
        f"Regression  datasets={cfg_reg.datasets}  models={cfg_reg.model_types}  "
        f"configs/optimizer={cfg_reg.num_samples}  epochs={cfg_reg.num_epochs}  seeds={cfg_reg.seeds}"
    )
    reg_results = run_experiments(cfg_reg)
    print("Regression complete")
else:
    reg_results = {}
    print("Regression skipped")


## ⑧ Run: Tabular Classification

In [ ]:
if RUN_TABULAR_CLASSIFICATION:
    defaults = TASK_DEFAULTS["tabular_classification"]
    tab_datasets = TABULAR_DATASETS or defaults["datasets"]
    tab_settings = task_run_settings("tabular_classification", tab_datasets)
    cfg_tab = ExperimentConfig(
        task_type   = "tabular_classification",
        datasets    = tab_datasets,
        model_types = TABULAR_MODELS or defaults["model_types"],
        num_samples = tab_settings["num_samples"],
        seeds       = tab_settings["seeds"],
        batch_size  = defaults["batch_size"],
        num_epochs  = tab_settings["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = tab_settings["mock_run"],
        output_dir  = str(OUT_DIR),
        mlflow_uri  = MFLOW_URI,
        data_dir    = str(DATA_DIR),
        num_workers = NUM_WORKERS,
    )
    print(
        f"Tabular CLS  datasets={cfg_tab.datasets}  models={cfg_tab.model_types}  "
        f"configs/optimizer={cfg_tab.num_samples}  epochs={cfg_tab.num_epochs}  seeds={cfg_tab.seeds}"
    )
    tab_results = run_experiments(cfg_tab)
    print("Tabular classification complete")
else:
    tab_results = {}
    print("Tabular classification skipped")


## ⑨ Run: Image Classification

In [ ]:
if RUN_IMAGE_CLASSIFICATION:
    defaults = TASK_DEFAULTS["image_classification"]
    image_datasets = IMAGE_DATASETS or defaults["datasets"]
    img_settings = task_run_settings("image_classification", image_datasets)
    cfg_img = ExperimentConfig(
        task_type   = "image_classification",
        datasets    = image_datasets,
        model_types = IMAGE_MODELS or defaults["model_types"],
        num_samples = img_settings["num_samples"],
        seeds       = img_settings["seeds"],
        batch_size  = defaults["batch_size"],
        num_epochs  = img_settings["num_epochs"],
        gpu_num     = GPU_NUM,
        mock_run    = img_settings["mock_run"],
        output_dir  = str(OUT_DIR),
        mlflow_uri  = MFLOW_URI,
        data_dir    = str(DATA_DIR),
        num_workers = NUM_WORKERS,
        kaggle_json = KAGGLE_JSON,
        grid_check = GRID_CHECK_IMAGE_PARAMS,
        grid_check_max_imgs = GRID_CHECK_MAX_IMGS,
    )
    print(
        f"Image CLS  datasets={cfg_img.datasets}  models={cfg_img.model_types}  "
        f"configs/optimizer={cfg_img.num_samples}  epochs={cfg_img.num_epochs}  seeds={cfg_img.seeds}"
    )
    img_results = run_experiments(cfg_img)
    print("Image classification complete")
else:
    img_results = {}
    print("Image classification skipped")


## Terminal / tmux launch with visible progress

If the repository is already cloned, start in that directory:

```bash
cd /root/optimizer_run/code  # or the directory that contains run_on_server.ipynb
python -m pip install --quiet --break-system-packages --root-user-action=ignore jupyterlab nbconvert
```

If this is a fresh server and the repository is not cloned yet:

```bash
mkdir -p /root/optimizer_run
cd /root/optimizer_run
git clone https://github.com/maximspbu/optimizers_comparison_tracking.git code
cd code
python -m pip install --quiet --break-system-packages --root-user-action=ignore jupyterlab nbconvert
```

### Option A - run from Jupyter in tmux

```bash
tmux new -s optimizers
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

Open `http://<instance-public-ip>:8888`, then run `run_on_server.ipynb` with `Kernel -> Restart & Run All`. Progress is visible in notebook cell output, MLflow, TensorBoard, and the log tail below.

### Option B - execute the notebook non-interactively in tmux

```bash
tmux new -s optimizers
PYTHONUNBUFFERED=1 jupyter nbconvert \
  --to notebook \
  --execute run_on_server.ipynb \
  --inplace \
  --ExecutePreprocessor.timeout=-1 \
  2>&1 | tee /root/optimizer_run/outputs/notebook_execute.log
```

In another terminal:

```bash
tmux attach -t optimizers
# or watch the runner log directly:
tail -f /root/optimizer_run/outputs/results/run.log
```

Detach from tmux without stopping the run: press `Ctrl-b`, then `d`. Reattach later with `tmux attach -t optimizers`.


## ⑩ Results

In [ ]:
import pandas as pd

results_dir = OUT_DIR / "results"

cross_csv = results_dir / "cross_summary.csv"
if cross_csv.exists():
    print("=== Cross-experiment summary ===")
    display(pd.read_csv(cross_csv, index_col=0))
else:
    print("cross_summary.csv not found yet")

In [ ]:
for csv_path in sorted(results_dir.glob("*_grid_check_summary.csv")):
    exp_key = csv_path.stem.replace("_grid_check_summary", "")
    print(f"\n=== {exp_key} — image grid check summary ===")
    display(pd.read_csv(csv_path))

    trials_csv = results_dir / f"{exp_key}_grid_check_trials.csv"
    if trials_csv.exists():
        print(f"\n=== {exp_key} — image grid check trials ===")
        display(pd.read_csv(trials_csv))

for csv_path in sorted(results_dir.glob("*_tuned_summary.csv")):
    exp_key = csv_path.stem.replace("_tuned_summary", "")
    print(f"\n=== {exp_key} — tuned ===")
    display(pd.read_csv(csv_path))

    default_csv = results_dir / f"{exp_key}_default_summary.csv"
    if default_csv.exists():
        print(f"\n=== {exp_key} — default ===")
        display(pd.read_csv(default_csv))

    compare_csv = results_dir / f"{exp_key}_tuned_vs_default.csv"
    if compare_csv.exists():
        print(f"\n=== {exp_key} — tuned vs default ===")
        display(pd.read_csv(compare_csv))

In [ ]:
import glob as _glob
from IPython.display import Image as IPImage, display as ipy_display

for subdir in ("best_run_plots", "tuned_run_plots", "default_run_plots", "varying_rs_run_plots"):
    pngs = sorted(_glob.glob(str(OUT_DIR / subdir / "*.png")))
    if not pngs:
        continue
    print(f"\n{'─'*60}  {subdir}  ({len(pngs)} plots)")
    for p in pngs:
        print(Path(p).name)
        ipy_display(IPImage(filename=p, width=900))

In [ ]:
from src.telemetry import load_telemetry, telemetry_summary

for jsonl in sorted((results_dir / "telemetry").glob("*.jsonl")):
    exp_key = jsonl.stem.replace("_telemetry", "")
    print(f"\n=== {exp_key} — telemetry ===")
    display(telemetry_summary(load_telemetry(str(jsonl))))

In [ ]:
# Start MLflow UI on port 5000
# Make sure port 5000 is open in your vast.ai instance network settings.
import subprocess as _sp
_sp.Popen(
    ["mlflow", "ui", "--backend-store-uri", MFLOW_URI,
     "--host", "0.0.0.0", "--port", "5000"],
    stdout=_sp.DEVNULL, stderr=_sp.DEVNULL,
)
import socket
hostname = socket.gethostname()
print(f"MLflow UI → http://<instance-public-ip>:5000")
print(f"(hostname: {hostname})")

In [ ]:
# Start TensorBoard on port 6006
# Make sure port 6006 is open in your vast.ai instance network settings.
import subprocess as _sp
_sp.Popen(
    ["tensorboard", "--logdir", str(OUT_DIR / "tensorboard"),
     "--host", "0.0.0.0", "--port", "6006"],
    stdout=_sp.DEVNULL, stderr=_sp.DEVNULL,
)
print(f"TensorBoard  → http://<instance-public-ip>:6006")
print(f"(logs appear after evaluation runs start writing data)")